In [ ]:
import os
import zipfile
from pathlib import Path
import json
import re
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.append(os.path.dirname(os.path.abspath("")))
from utils.utils import set_seed, get_experiment_id, get_run_dir_full
# set_seed(42)

In [ ]:
config_file_name_list = ["config_crc_trainP5_valP1.json", "config_crc_trainP5V_valP1.json", "config_crc_trainP5_valP2.json", "config_crc_trainP5V_valP2.json", "config_crc_trainP5_valP5.json", "config_crc_trainP5V_valP5.json"]
fold_id = 1
epoch = "last"  # "last" or specific epoch number (starting from 0)
gpu_id = 3

Dataset:

- Manuscript: https://www.nature.com/articles/s41588-025-02193-3#data-availability
- 10x: https://www.10xgenomics.com/platforms/visium/product-family/dataset-human-crc
- 10x (alt): https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc
- GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280318
- SRA: https://www.ncbi.nlm.nih.gov/Traces/study/?acc=PRJNA1177833&o=acc_s%3Aa

In [ ]:
os.chdir("../")

## Do more validation

In [ ]:
for i, config_file_name in enumerate(config_file_name_list):
    print(f"Running validation for file {i+1}/{len(config_file_name_list)}: {config_file_name}...")
    run_dir_full = get_run_dir_full(config_file_name, fold_id)
    validation_set = config_file_name.split("_")[-1].split(".")[0][-2:]
    val_log_path = f"{run_dir_full}/val_{validation_set}_out.txt"
    
    !python inference.py --config_file configs/{config_file_name} --epoch {epoch} --mode val --fold_id {fold_id} --gpu_id {gpu_id} | tee {val_log_path}

    def extract_metrics(log_path, pattern):
        """Return {epoch: value} from log file matching given regex pattern."""
        results = {}
        try:
            with open(log_path) as f:
                for line in f:
                    match = pattern.search(line)
                    if match:
                        epoch = int(match.group(1))
                        value = float(match.group(2))
                        results[epoch] = value
        except FileNotFoundError:
            print(f"File not found: {log_path}")
        return results

    metrics_config = [
        {
            "split": "train",
            "metric": "loss",
            "file": train_log_path,
            "regex": re.compile(r"Epoch\[(\d+)/\d+\],\s*loss:([0-9]*\.?[0-9]+)", re.IGNORECASE),
        },
        {
            "split": "val",
            "metric": "expr_pcc",
            "file": val_log_path,
            "regex": re.compile(r"Epoch\[(\d+)\],\s*PCC\s+mean\s*\(expression(?:\scorrelation)?\):\s*([0-9]*\.?[0-9]+)", re.IGNORECASE),
        },
        {
            "split": "val",
            "metric": "celltype_f1",
            "file": val_log_path,
            "regex": re.compile(r"Epoch\[(\d+)\],\s*F1\s*\(cell types\):\s*([0-9]*\.?[0-9]+)", re.IGNORECASE),
        },
        {
            "split": "val",
            "metric": "variant_f1",
            "file": val_log_path,
            "regex": re.compile(r"Epoch\[(\d+)\],\s*F1\s*\(variants\):\s*([0-9]*\.?[0-9]+)", re.IGNORECASE),
        },
    ]

    df_metrics = pd.DataFrame(metrics_config)
    df_metrics["dict"] = df_metrics.apply(lambda row: extract_metrics(row["file"], row["regex"]), axis=1)

    df_metrics

## Visualize results